# VectorDrugMatch - Colab training
Runtime > Change runtime type > **T4 GPU**, then Run all. Raw Hetionet data is downloaded and preprocessed inside Colab, so no data upload is needed.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
!git clone https://github.com/MUKUL-PRASAD-SIGH/VectorDrugMatch
%cd VectorDrugMatch
!git checkout mukul

In [ ]:
!pip install -q rdkit mlflow optuna shap xgboost lightgbm loguru pyyaml torch-geometric
import torch
v=torch.__version__.split('+')[0]; c=(torch.version.cuda or '').replace('.','')
!pip install -q pyg_lib torch_scatter torch_sparse torch_cluster -f https://data.pyg.org/whl/torch-{v}+cu{c}.html
print(torch.__version__, torch.cuda.is_available())

In [ ]:
# Download full Hetionet (nodes + edges) and build data/processed/baseline_morgan
# SMILES come from the MyChem API (PubChem CID-SMILES is optional and ~GBs, skipped).
# Optional: put DisGeNET's curated_gene_disease_associations.tsv in data/raw/ first.
!mkdir -p data/raw
!rm -f data/raw/drugs.csv data/raw/diseases.csv data/raw/genes.csv data/raw/edges.csv  # drop toy mock data
B='https://github.com/hetio/hetionet/raw/main/hetnet/tsv'
!wget -q -P data/raw $B/hetionet-v1.0-nodes.tsv $B/hetionet-v1.0-edges.sif.gz
!python scripts/preprocess.py --config configs/baseline_morgan.yaml
!cat data/processed/graph_metadata.json
!wc -l data/processed/baseline_morgan/*.csv
# Save processed data to Drive so it can be reused without re-running
!mkdir -p /content/drive/MyDrive/VDM_Data && cp -r data/processed/baseline_morgan /content/drive/MyDrive/VDM_Data/

In [ ]:
CFG='configs/baseline_morgan.yaml'
!python scripts/train_baselines.py --model all --config $CFG
!python scripts/train_gnn.py --config $CFG
!python scripts/train_benchmark.py --config $CFG

In [ ]:
# Optional
# !python scripts/run_ablations.py --config $CFG
# !python scripts/optuna_gnn.py --config $CFG --trials 20

In [ ]:
import pandas as pd
out='outputs/baseline_morgan'
display(pd.read_csv(f'{out}/baseline_comparison.csv'))
display(pd.read_csv(f'{out}/gnn_comparison.csv'))
!ls {out}/models

In [ ]:
!cp -r outputs /content/drive/MyDrive/VDM_Outputs_Backup
# Download trained models + metrics as a zip
!zip -rq vdm_results.zip outputs
from google.colab import files
files.download('vdm_results.zip')